In [ ]:
from IPython.display import display, HTML

display(HTML("""
<style>

/* =========================
전체 레이아웃
========================= */

div.container{
    width:85% !important;
}

div.cell.code_cell.rendered{
    width:100%;
}

div.input_prompt{
    padding:0;
}

div.prompt{
    min-width:70px;
}

div#toc-wrapper{
    padding-top:120px;
}

table.dataframe{
    font-size:12px;
}

/* =========================
   코드 입력창
========================= */

div.CodeMirror{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
    line-height:1.6;
}

/* =========================
   입력 셀
========================= */

div.input{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   코드 출력
========================= */

div.output{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   Markdown 전체
========================= */

.rendered_html{
    font-family:"마루 부리OTF 중간" !important;
    font-size:18px !important;
    line-height:1.8;
}

/* 제목 */

.rendered_html h1,
.rendered_html h2,
.rendered_html h3,
.rendered_html h4,
.rendered_html h5,
.rendered_html h6{
    font-family:"마루 부리OTF 조금굵은" !important;
}

/* 본문 */

.rendered_html p{
    font-family:"마루 부리OTF 중간" !important;
}

/* 리스트 */

.rendered_html li{
    font-family:"마루 부리OTF 중간" !important;
    padding:5px;
}

/* 인용 */

.rendered_html blockquote{
    font-family:"마루 부리OTF 중간" !important;
}

/* 표 */

.rendered_html table{
    font-family:"마루 부리OTF 중간" !important;
}

/* 코드 블록 */

.rendered_html pre,
.rendered_html code{
    font-family:"Consolas" !important;
    font-size:12pt !important;
}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 벡터 DB : Chroma vs. Pinecone
- Chroma : 로컬 vector DB, collection 단위로 관리
- Pinecone : 클라우드 vector DB, index 단위로 관리

## 내 Pinecone 설정으로 실행하기
1. `C:\ai\source\07_LLM\.env`에 `OPENAI_API_KEY`, `PINECONE_API_KEY`, `UPSTAGE_API_KEY`를 설정합니다. 키는 노트북에 직접 입력하지 않습니다.
2. 설치 셀을 실행합니다. 이미 관련 라이브러리를 불러온 상태라면 설치 후 커널을 재시작하고 위에서부터 실행합니다.
3. 기존 `tax-index`의 HOST로 연결하고 OpenAI `text-embedding-3-large`의 출력 차원을 **1024**로 지정합니다.
4. 콘솔의 `llama-text-embed-v2`와 OpenAI는 다른 임베딩 모델입니다. OpenAI 문서는 전용 namespace `openai-3-large-1024-v1`에 새로 저장하고 같은 모델로 검색합니다. 이 노트북은 Pinecone의 자동 텍스트 임베딩 대신 직접 생성한 벡터를 사용합니다.
5. 전용 namespace가 비어 있으면 첫 실행 시 문서를 임베딩해 업로드합니다(API 사용량 발생). 이후에는 저장된 벡터를 사용합니다. 기존 namespace의 데이터는 삭제하지 않습니다.

공식 문서: [OpenAI 임베딩 차원](https://developers.openai.com/api/docs/guides/embeddings), [Pinecone 벡터 업로드](https://docs.pinecone.io/guides/index-data/upsert-data)

HOST, 인덱스, namespace, 임베딩 모델/차원도 위 `.env`에서 읽습니다. `PINECONE_API_KEY`는 기존 `.env`에 저장되어 있습니다. 키 값은 노트북에 기록하지 않습니다. 수정한 `.env`는 공통 연결 설정 셀을 다시 실행하면 반영됩니다.

API 키 이름은 콘솔에서 관리하는 표시 이름입니다. 코드에는 키 이름 대신 `.env`의 실제 `PINECONE_API_KEY` 값을 사용합니다.


# 0. 패키지 설치

In [ ]:
%pip install -q pinecone langchain-pinecone langchain-openai langchain-upstage langchain-community langchain-text-splitters python-dotenv docx2txt


# 1. knowledge base 구성을 위한 데이터 생성

In [ ]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

In [ ]:
# 공통 연결 설정: 아래 설정을 저장/검색/RAG 전체에서 함께 사용합니다.
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

# 실행 위치와 관계없이 사용자가 지정한 기존 .env를 읽습니다.
env_path = Path(r"C:\ai\source\07_LLM\.env")
if not env_path.is_file():
    raise FileNotFoundError(f"환경 설정 파일이 없습니다: {env_path}")
load_dotenv(env_path, override=True)

required_keys = ("OPENAI_API_KEY", "PINECONE_API_KEY", "UPSTAGE_API_KEY")
missing_keys = [name for name in required_keys if not os.getenv(name, "").strip()]
if missing_keys:
    raise ValueError(".env에 다음 키를 설정한 후 다시 실행하세요: " + ", ".join(missing_keys))

PINECONE_INDEX_NAME = os.environ["PINECONE_INDEX_NAME"].strip()
PINECONE_HOST = os.environ["PINECONE_HOST"].strip()
PINECONE_NAMESPACE = os.environ["PINECONE_NAMESPACE"].strip()
EMBEDDING_MODEL = os.environ["EMBEDDING_MODEL"].strip()
EMBEDDING_DIMENSIONS = int(os.environ["EMBEDDING_DIMENSIONS"])

embedding = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    dimensions=EMBEDDING_DIMENSIONS,
)

print(f"설정 준비 완료: {PINECONE_INDEX_NAME} / {EMBEDDING_MODEL} / {EMBEDDING_DIMENSIONS}차원")


In [ ]:
# # 임베딩 객체
# from dotenv import load_dotenv
# from langchain_upstage import UpstageEmbeddings
# load_dotenv()
# embedding = UpstageEmbeddings(model = 'solar-embedding-1-large-passage')

In [ ]:
# 실제 OpenAI 호출로 질문 벡터의 길이를 확인합니다(API 사용량 발생).
query_dimension = len(embedding.embed_query("안녕"))
assert query_dimension == EMBEDDING_DIMENSIONS, f"임베딩 차원 불일치: {query_dimension}"
print(f"임베딩 차원: {query_dimension}")


In [ ]:
# 모델마다 코사인유사도를 쓰는지 document
# https://console.upstage.ai/docs/capabilities/embed
# https://developers.openai.com/api/docs/guides/embeddings

In [ ]:
%%time
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from hashlib import sha256
import time

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
# 인덱스 이름 조회 대신 사용자가 제공한 HOST로 직접 연결합니다.
index_name = PINECONE_INDEX_NAME
index = pc.Index(host=PINECONE_HOST)
stats = index.describe_index_stats()
if stats["dimension"] != EMBEDDING_DIMENSIONS:
    raise ValueError(f"인덱스 차원 {stats['dimension']} != 임베딩 차원 {EMBEDDING_DIMENSIONS}")

database = PineconeVectorStore(
    index=index,
    embedding=embedding,
    namespace=PINECONE_NAMESPACE,
    text_key="text",
)

# 첫 실행에는 문서를 저장해야 검색할 수 있습니다.
# 전용 namespace만 확인하며, 다른 namespace는 수정하지 않습니다.
namespace_count = stats.get("namespaces", {}).get(PINECONE_NAMESPACE, {}).get("vector_count", 0)
if namespace_count == 0:
    if not documents:
        raise ValueError("읽어 온 문서가 없습니다. 문서 로딩 셀부터 확인하세요.")
    # 같은 문서를 재실행해도 동일 ID를 사용합니다.
    document_ids = [
        sha256(f"{i}:{doc.metadata.get('source', '')}:{doc.page_content}".encode("utf-8")).hexdigest()
        for i, doc in enumerate(documents)
    ]
    database.add_documents(documents=documents, ids=document_ids, batch_size=64)
    # Pinecone은 업로드 직후 검색 반영까지 시간이 걸릴 수 있습니다.
    deadline = time.monotonic() + 60
    while True:
        current_stats = index.describe_index_stats()
        current_count = current_stats.get("namespaces", {}).get(PINECONE_NAMESPACE, {}).get("vector_count", 0)
        if current_count >= len(documents):
            break
        if time.monotonic() >= deadline:
            raise TimeoutError("업로드 후 반영 대기 시간이 지났습니다. 잠시 후 연결 셀을 다시 실행하세요.")
        time.sleep(2)
    print(f"최초 업로드 완료: {len(documents)}개 청크")
else:
    if namespace_count != len(documents):
        raise ValueError(
            f"저장된 청크({namespace_count})와 현재 문서 청크({len(documents)}) 수가 다릅니다. "
            "공통 설정의 namespace를 새로운 이름으로 바꾼 후 다시 실행하세요."
        )
    print(f"기존 벡터 사용: {namespace_count}개 청크")
print(f"연결: {index_name} / namespace: {PINECONE_NAMESPACE}")


# 2. 답변 생성을 위한 Retrieval

In [ ]:
query = "연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query, k=3) # 기본k값은 4

In [ ]:
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n','\n') for doc in retrieved_docs])
#print(retrieved_doc)

# 3. 답변생성

In [ ]:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model='gpt-4.1-nano')

In [ ]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model = 'solar-pro2')

In [ ]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다. [context]를 참고해서 사용자 질문에 답하세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [ ]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

# 4. Langchain

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_upstage import ChatUpstage
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 앞에서 만든 LLM과 OpenAI 임베딩 설정을 그대로 사용합니다.
# 위의 문서 로딩/공통 설정/Pinecone 연결/LLM 셀부터 실행하세요.
# 2. 같은 HOST, namespace, 임베딩 객체를 사용합니다.
vectorStore = database
# 3. Retriever 생성
# (1) search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity', 
                                     search_kwargs={'k':3}) # 유사도가 높은 top-3추출
# # (2) search_type='mmr' : 후보 10개 중 서로 덜 겹치는 3개 추출
# retriever = vectorStore.as_retriever(search_type='mmr', 
#                                      search_kwargs={'k':3,
#                                                    'fetch_k':10,
#                                                    'lambda_mult':0.5}) # 1에 가까울수록 관련성 깊은 계산
# # (3) search_type='similarity_score_threshold' : 
# retriever = vectorStore.as_retriever(search_type='similarity_score_threshold', 
#                                      search_kwargs={'k':3, 'score_threshold':0.4}) 
# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)